# 13.3 视频语言 (Video-Language)

视频语言模型在视觉语言基础上增加时序维度，理解视频中的动态信息、动作序列和事件发展。

## 视频语言模型的关键挑战

| 挑战 | 描述 | 解决方案 |
|------|------|----------|
| 时序建模 | 理解帧间动态变化 | 时序注意力/3D卷积 |
| 计算量 | 视频帧数多，计算量大 | 帧采样/Token压缩 |
| 长视频 | 长视频超出上下文窗口 | 分段处理+摘要 |
| 音频视觉融合 | 视频包含音视频双模态 | 多模态融合 |

## 1. 视频编码器与时序建模

视频编码器需要同时捕获空间特征（每帧内容）和时间特征（帧间变化）。

### 帧采样策略
- **均匀采样**：从视频中均匀选取N帧，最简单
- **关键帧采样**：选择场景变化的关键帧，信息密度更高
- **自适应采样**：根据内容动态调整采样率

### 时序建模方法
1. **后期融合**：每帧独立编码，最后聚合
2. **时序注意力**：帧特征间自注意力
3. **时空注意力**：同时建模空间和时间关系
4. **3D卷积**：直接在时空维度上卷积

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class VideoEncoder(nn.Module):
    def __init__(self, d_frame=64, d_video=128, n_frames=8, n_heads=2):
        super().__init__()
        self.frame_encoder = nn.Sequential(
            nn.Linear(3 * 16 * 16, 128), nn.ReLU(), nn.Linear(128, d_frame)
        )
        self.temporal_pos = nn.Parameter(torch.randn(1, n_frames, d_frame) * 0.02)
        self.temporal_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_frame, n_heads, d_frame * 4, batch_first=True), 2
        )
        self.proj = nn.Linear(d_frame, d_video)

    def forward(self, frames):
        B, T, C, H, W = frames.shape
        frame_features = self.frame_encoder(frames.reshape(B, T, -1))
        frame_features = frame_features + self.temporal_pos[:, :T, :]
        temporal_features = self.temporal_encoder(frame_features)
        return self.proj(temporal_features)

class SpatialTemporalEncoder(nn.Module):
    def __init__(self, d_model=64, n_heads=2, n_frames=8):
        super().__init__()
        self.patch_embed = nn.Linear(3 * 4 * 4, d_model)
        n_patches = (16 // 4) ** 2
        self.spatial_pos = nn.Parameter(torch.randn(1, n_patches, d_model) * 0.02)
        self.temporal_pos = nn.Parameter(torch.randn(1, n_frames, d_model) * 0.02)
        self.st_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, n_heads, d_model * 4, batch_first=True), 2
        )

    def forward(self, frames):
        B, T, C, H, W = frames.shape
        p = 4
        patches = frames.reshape(B, T, C, H//p, p, W//p, p)
        patches = patches.permute(0, 1, 3, 5, 2, 4, 6).reshape(B, T, (H//p)*(W//p), -1)
        tokens = self.patch_embed(patches)
        tokens = tokens + self.spatial_pos.unsqueeze(1)
        tokens = tokens + self.temporal_pos.unsqueeze(2)
        tokens = tokens.reshape(B, -1, tokens.shape[-1])
        tokens = self.st_encoder(tokens)
        return tokens

frames = torch.randn(2, 8, 3, 16, 16)

video_enc = VideoEncoder()
video_features = video_enc(frames)

st_enc = SpatialTemporalEncoder()
st_features = st_enc(frames)

print('=== Video Encoding ===')
print(f'Input: {frames.shape} (batch, frames, C, H, W)')
print(f'\nTemporal encoder output: {video_features.shape}')
print(f'  Per-frame features with temporal context')
print(f'\nSpatial-Temporal encoder output: {st_features.shape}')
print(f'  All spatial-temporal tokens flattened')

print(f'\nKey: Temporal encoder adds temporal position and attention across frames.')
print(f'Spatial-Temporal encoder jointly models spatial patches and temporal frames.')

## 2. 视频语言模型

视频语言模型将视频特征与语言模型结合，支持视频问答、视频摘要、视频描述等任务。

### 代表模型
| 模型 | 视频编码 | 帧数 | 特点 |
|------|----------|------|------|
| Video-LLaVA | CLIP ViT | 8 | 简单投影式 |
| LLaMA-VID | CLIP ViT | 变长 | 每帧一个token压缩 |
| VideoChat2 | CLIP ViT | 16 | 时序感知投影 |
| GPT-4V | 原生 | 动态 | 统一多模态 |

### Token压缩策略
视频帧数多导致token数量爆炸，需要压缩：
- **空间池化**：2×2池化，4个token→1个
- **时序池化**：相邻帧平均，减少帧数
- **Q-Former**：用查询向量压缩视觉特征
- **每帧一token**：最激进的压缩

In [ ]:
class VideoLLM(nn.Module):
    def __init__(self, d_video=128, d_language=128, vocab_size=500,
                 n_frames=8, compression='spatial_pool'):
        super().__init__()
        self.video_encoder = VideoEncoder(d_frame=64, d_video=d_video, n_frames=n_frames)
        self.compression = compression
        self.video_proj = nn.Linear(d_video, d_language)
        self.text_embed = nn.Embedding(vocab_size, d_language)
        self.lm = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_language, 2, d_language * 4, batch_first=True), 2
        )
        self.head = nn.Linear(d_language, vocab_size)

    def compress_tokens(self, video_tokens):
        B, T, D = video_tokens.shape
        if self.compression == 'spatial_pool':
            if T > 4:
                video_tokens = video_tokens.reshape(B, T // 2, 2, D).mean(dim=2)
        elif self.compression == 'temporal_pool':
            video_tokens = video_tokens.mean(dim=1, keepdim=True)
        elif self.compression == 'none':
            pass
        return video_tokens

    def forward(self, frames, text_tokens):
        video_features = self.video_encoder(frames)
        compressed = self.compress_tokens(video_features)
        vis_tokens = self.video_proj(compressed)
        txt_tokens = self.text_embed(text_tokens)
        combined = torch.cat([vis_tokens, txt_tokens], dim=1)
        h = self.lm(combined)
        text_output = h[:, vis_tokens.shape[1]:]
        return self.head(text_output)

frames = torch.randn(2, 8, 3, 16, 16)
text = torch.randint(0, 500, (2, 10))

print('=== Video LLM with Token Compression ===')
for comp in ['none', 'spatial_pool', 'temporal_pool']:
    model = VideoLLM(compression=comp)
    out = model(frames, text)
    n_vis = sum(p.numel() for p in model.video_proj.parameters())
    print(f'{comp:15s}: output={out.shape}, vis_tokens after compression varies')

print(f'\nKey: Token compression is critical for video LLMs.')
print(f'Spatial pool halves tokens. Temporal pool compresses to 1 token per video.')

## 3. 视频语言模型工业实践

### 帧数选择
| 帧数 | 视频长度(30fps) | 计算量 | 适用场景 |
|------|-----------------|--------|----------|
| 4帧 | ~4秒 | 低 | 短视频分类 |
| 8帧 | ~8秒 | 中 | 标准视频问答 |
| 16帧 | ~16秒 | 高 | 详细视频理解 |
| 32帧 | ~32秒 | 很高 | 长视频分析 |

### 最佳实践
1. **帧采样**：8帧是性价比最高的选择
2. **分辨率**：224×224足够，高分辨率收益有限
3. **Token压缩**：必须压缩，否则超出上下文窗口
4. **训练策略**：先用图像数据预训练，再在视频数据微调
5. **长视频处理**：分段编码+时序摘要+全局聚合
6. **音频融合**：视频中的音频信息很重要，不要忽略